# MS2: Data Wrangling & Project Redefinition — TRACE Benchmark

**Project:** Detecting Reward Hacking in AI Agent Trajectories  
**Dataset:** 
- [PatronusAI/trace-dataset](https://huggingface.co/datasets/PatronusAI/trace-dataset) — 517 labeled trajectories  
- [metr-evals/malt-public](https://huggingface.co/datasets/metr-evals/malt-public) — ~10K+ trajectories
- [Jozdien/realistic_reward_hacks](https://huggingface.co/datasets/Jozdien/realistic_reward_hacks) — 100+ trajectories

**Canvas Project #:** _TODO_  
**Group Members:** _TODO_

In [106]:
import json
import os
import collections
import dotenv
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk, load_dataset

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Data Acquisition & Loading

**Source:** TRACE benchmark from Patronus AI on Hugging Face (gated dataset).  
Downloaded via `datasets` library and saved locally as Arrow format.

Need to get Hugging Face token for access (set as env var or input at runtime). After getting a token, you need to request access to both datasets on Hugging Face:
- TRACE: https://huggingface.co/datasets/PatronusAI/trace-dataset
- MALT: https://huggingface.co/datasets/metr-evals/malt-public  
- REALISTIC_REWARD_HACKS: https://huggingface.co/datasets/Jozdien/realistic_reward_hacks


In [107]:
dotenv.load_dotenv()  # Load .env if it exists, to get HF token from environment variables

# update ROOT to be the parent of the current file's directory, which is more robust than hardcoding the path
ROOT = Path.home() / "Harvard" / "CSCIE109B" / "project" / "reward-hacking"

# relative paths for data directories
TRACE_DIR = ROOT / "data" / "trace"
MALT_DIR = ROOT / "data" / "malt" / "public"
REALISTIC_REWARD_HACKS_DIR = ROOT / "data" / "realistic_reward_hacks"


def get_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("hugging_face_token")
    if not token:
       # ask for token input if not found in env vars or .env file
        token = input("Enter your Hugging Face token: ").strip()
    return token

# Download and save datasets locally
def download_trace():
    print("Downloading TRACE dataset...")
    ds = load_dataset("PatronusAI/trace-dataset", token=get_token())
    TRACE_DIR.mkdir(parents=True, exist_ok=True)
    ds["train"].save_to_disk(str(TRACE_DIR))
    print(f"Saved {ds['train'].num_rows} TRACE trajectories to {TRACE_DIR}")

# MALT is a larger dataset, so we will save each split separately
def download_malt():
    print("Downloading MALT dataset...")
    ds = load_dataset("metr-evals/malt-public", token=get_token())
    MALT_DIR.mkdir(parents=True, exist_ok=True)
    for split in ds:
        out = MALT_DIR / split
        ds[split].save_to_disk(str(out))
        print(f"Saved {ds[split].num_rows} MALT rows ({split}) to {out}")

# The REALISTIC_REWARD_HACKS dataset is smaller, so we can save it all together
def download_realistic_reward_hacks():
    print("Downloading REALISTIC_REWARD_HACKS dataset...")
    ds = load_dataset("Jozdien/realistic_reward_hacks", token=get_token())
    REALISTIC_REWARD_HACKS_DIR.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(REALISTIC_REWARD_HACKS_DIR))
    print(f"Saved {ds.num_rows} REALISTIC_REWARD_HACKS trajectories to {REALISTIC_REWARD_HACKS_DIR}")

In [108]:
# check if trace and malt datasets already exist locally before downloading
if os.path.exists(TRACE_DIR) and os.path.exists(MALT_DIR) and os.path.exists(REALISTIC_REWARD_HACKS_DIR):
    
    print(f"Loading datasets from disk...")
    ds_trace = load_from_disk(TRACE_DIR)
    ds_malt = load_from_disk(MALT_DIR)
    ds_realistic_reward_hacks = load_from_disk(REALISTIC_REWARD_HACKS_DIR)
else:
    print("TRACE dataset not found locally. Downloading...")
    download_trace()
    print("MALT dataset not found locally. Downloading...")
    download_malt()
    print("REALISTIC_REWARD_HACKS dataset not found locally. Downloading...")
    download_realistic_reward_hacks()
    



Loading datasets from disk...


Loading dataset from disk:   0%|          | 0/42 [00:00<?, ?it/s]

In [109]:
def load_and_report_dataset(dataset_dir):
    ds = load_from_disk(dataset_dir)
    data_size_mb = sum(
        os.path.getsize(os.path.join(dataset_dir, f))
        for f in os.listdir(dataset_dir)
    ) / 1024 / 1024
    print(f"Dataset: {ds}")
    print(f"Features: {ds.features}")
    print(f"Rows: {ds.num_rows}")
    print(f"Size on disk: {data_size_mb:.1f} MB")
    return ds

In [110]:
TRACE_DIR = "../data/trace"
ds_trace = load_and_report_dataset(TRACE_DIR)

Dataset: Dataset({
    features: ['trajectory_id', 'conversation', 'label'],
    num_rows: 517
})
Features: {'trajectory_id': Value('string'), 'conversation': Value('string'), 'label': Value('string')}
Rows: 517
Size on disk: 21.7 MB


In [111]:
MALT_DATASET_PATH = "../data/malt/public"
ds_malt = load_and_report_dataset(MALT_DATASET_PATH)

Loading dataset from disk:   0%|          | 0/42 [00:00<?, ?it/s]

Dataset: Dataset({
    features: ['samples', 'metadata'],
    num_rows: 7179
})
Features: {'samples': LargeList({'input': LargeList({'role': Value('string'), 'content': Value('large_string'), 'name': Value('string'), 'function_call': {'name': Value('string'), 'arguments': Value('string')}, 'metadata': {'completion_type': Value('string'), 'contains_function_call': Value('bool'), 'function_call_result': Value('bool'), 'node_id': Value('int64'), 'parent_node_id': Value('int64'), 'branch_id': Value('int64'), 'timestamp': Value('int64')}}), 'output': LargeList(LargeList({'role': Value('string'), 'content': Value('large_string'), 'name': Value('null'), 'function_call': {'name': Value('string'), 'arguments': Value('string')}, 'metadata': {'completion_type': Value('string'), 'contains_function_call': Value('null'), 'function_call_result': Value('null'), 'node_id': Value('null'), 'parent_node_id': Value('null'), 'branch_id': Value('null'), 'timestamp': Value('null')}})), 'metadata': {'unmatched

In [112]:
# we can use both the litarary and code reward hacks, since they are just different splits of the same dataset
REALISTIC_REWARD_HACKS_DATASET_PATH = "../data/realistic_reward_hacks/combined"
ds_realistic_reward_hacks = load_and_report_dataset(REALISTIC_REWARD_HACKS_DATASET_PATH)


Dataset: Dataset({
    features: ['messages'],
    num_rows: 1605
})
Features: {'messages': List({'content': Value('string'), 'role': Value('string')})}
Rows: 1605
Size on disk: 6.5 MB


## 2. Data Understanding & Summary

#### TRACE Dataset

In [113]:
df_trace = ds_trace.to_pandas()
display(df_trace.head())

,trajectory_id,conversation,label
0,trajectory_0389,"[{""role"": ""user"", ""content"": ""Our WebSocket se...",0
1,trajectory_0223,"[{""role"": ""user"", ""content"": ""Refactor our ETL...",0
2,trajectory_0366,"[{""role"": ""user"", ""content"": ""Our live encoder...",0
3,trajectory_0448,"[{""role"": ""user"", ""content"": ""Hey, our CI pipe...",0
4,trajectory_0018,"[{""role"": ""user"", ""content"": ""We have a SOC2 a...",1.2.3


In [114]:
df_trace.info()

<class 'pandas.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   trajectory_id  517 non-null    str  
 1   conversation   517 non-null    str  
 2   label          517 non-null    str  
dtypes: str(3)
memory usage: 21.7 MB


In [115]:
unique_labels = set()
for label_str in df_trace["label"]:
    if label_str != "0":  # skip benign trajectories
        codes = [code.strip() for code in label_str.split(",")]
        unique_labels.update(codes)

print(f"Unique reward hack category codes in TRACE dataset: {sorted(unique_labels)}")

Unique reward hack category codes in TRACE dataset: ['1.1.1', '1.1.2', '1.1.3', '1.2.1', '1.2.2', '1.2.3', '1.3.1', '1.3.2', '1.4.1', '1.4.2']


In [116]:
conv = df_trace.loc[0, "conversation"]
df_conv = pd.json_normalize(json.loads(conv))
print(df_conv.columns)

Index(['role', 'content', 'tool_calls', 'tool_results'], dtype='str')


#### MALT Dataset


In [117]:
# Extract just metadata — fast and small
df_malt = ds_malt.select_columns(["metadata"]).to_pandas()

# Flatten the metadata dict column
df_malt_meta = pd.json_normalize(df_malt["metadata"])

# Now you have: labels, model, task_id, run_id, run_source, etc.
print(f" MALT metadata columns: {df_malt_meta.columns.tolist()}")
print(f"Unique reward hack category codes in MALT dataset: {df_malt_meta['labels'].explode().unique()}")
print(f"Number of unique reward hack category codes in MALT dataset: {df_malt_meta['labels'].explode().nunique()}")


 MALT metadata columns: ['has_chain_of_thought', 'labels', 'manually_reviewed', 'model', 'public', 'run_id', 'run_source', 'task_id']
Unique reward hack category codes in MALT dataset: <ArrowStringArray>
[                 'gives_up',                    'normal',
        'bypass_constraints',        'hardcoded_solution',
                  'sabotage',      'reasoning_about_task',
   'partial_problem_solving',                  'refusals',
        'match_weaker_model', 'ignores_task_instructions']
Length: 10, dtype: str
Number of unique reward hack category codes in MALT dataset: 10


 Strict reward hacking only (matches TRACE most closely):

Benign: normal
Hacked: sabotage, bypass_constraints, hardcoded_solution, ignores_task_instructions, reasoning_about_task
Drop: refusals, partial_problem_solving, match_weaker_model, gives_up (these are "sandbagging" — deliberate underperformance, not reward hacking)

#### REALISTIC_REWARD_HACKS Dataset

### 2.1 Missing Data Analysis

TRACE is a curated benchmark, so we expect minimal missingness. We check all fields to confirm and classify any missingness pattern (MCAR/MAR/MNAR).

### 2.2 Class Imbalance Assessment

## 3. Feature Extraction & Preprocessing

We extract structural and behavioral features from the raw conversation JSON to enable analysis and future modeling. Each preprocessing step is documented and justified below.

### 3.1 Feature Scaling Assessment

We assess whether numeric features need scaling for future modeling. This is relevant for distance-based models (SVM, k-NN) but not tree-based models (XGBoost).

### 3.2 Anomaly Detection

We check for anomalous trajectories — those with unusually short/long conversations or extreme turn counts — using the IQR method on key numeric features (`num_turns`, `conversation_len_chars`, `n_tool_calls`).

Since TRACE is a curated benchmark, we expect any outliers to be valid long/short trajectories rather than data errors. If confirmed, they will be retained as natural variance in trajectory complexity.

## 4. EDA & Visualizations

Clean, labeled visualizations exploring the structure and patterns in the TRACE dataset.

### 4.1 Distribution of Numeric Features

Histograms of `num_turns`, `conversation_len_chars`, `n_tool_calls`, and `n_user` split by class (benign vs. hacked) to assess whether any single structural feature separates the two classes visually.

### 4.2 Correlation Analysis

A correlation heatmap across all numeric features and the binary label. This reveals whether any simple linear relationships exist between features, and how strongly each feature correlates with the hack label.

### 4.3 Hack Category Breakdown

Bar chart showing the distribution of fine-grained hack category codes among the 268 hacked trajectories. Some trajectories carry multiple category annotations. This visualization reveals which hack types are most prevalent and whether subcategory imbalance exists (relevant if we extend to multi-label classification).

### 4.4 Conversation Length vs. Turns

Scatter plot of conversation length (chars) vs. number of turns, colored by class. This helps identify whether hacked trajectories cluster in a distinct region of this feature space, or whether the two classes overlap substantially.

## 5. Meaningful Insights

Based on the EDA above, we identify the following insights that directly inform our modeling decisions:

1. **Balanced classes simplify modeling.** The near 50/50 split (268 hacked vs 249 benign) means we do not need resampling techniques like SMOTE. Standard cross-entropy loss is appropriate.

2. **Conversation length and turn count vary widely.** Trajectories range from ~11K to ~95K characters and 14-49 turns. This confirms that trajectory length is a real constraint for Transformer-based models — many examples will exceed standard 512-token context windows, requiring truncation or chunking strategies.

3. **Structural features alone may have limited discriminative power.** The correlation analysis shows weak linear relationships between basic features (turn count, conversation length, tool calls) and the hack label. This suggests that the *content* of turns — not just their count — carries the signal, supporting the use of a Transformer encoder that reads the full sequence.

4. **Hack categories are not uniformly distributed.** Some fine-grained categories appear much more frequently than others. If we extend to multi-label classification, this imbalance across subcategories will need to be addressed (unlike the balanced binary case).

5. **No missing data or obvious data errors.** The dataset is clean and curated, which is expected for a benchmark. Any outliers detected are valid long/short trajectories, not noise.

## 6. Summary of Findings

| Aspect | Finding |
|--------|---------|
| **Dataset size** |  |
| **Class balance** |  |
| **Missing data** | |
| **Anomalies** ||
| **Feature scales** | |
| **Key challenge** |  |
| **Signal source** | |

## 7. Rescoped Research Question

**Original question (MS0):** *Can we detect reward hacking from the trajectory alone?*

**Refined question (MS2):** *Can a fine-tuned Transformer encoder (e.g., RoBERTa) classify AI agent trajectories as reward-hacked or benign using the full conversation trace, and which structural or content-based features contribute most to detection?*

**Refinements based on EDA:**
- We confirmed that simple structural features (turn count, length) are weakly correlated with the label, motivating a model that reads full text rather than hand-crafted features alone.
- The balanced class distribution confirms binary classification as the primary task, with multi-label subcategory prediction as a stretch goal.
- Token length analysis will guide our truncation/chunking strategy in the next milestone.

## 8. Next Steps

**Delineation of tasks for upcoming milestones:**

1. **Tokenization & length analysis:** Tokenize trajectories with RoBERTa tokenizer, analyze token-length distribution, and design a truncation/chunking strategy for examples exceeding the 512-token limit.
2. **Train/val/test split:** Create an 80/10/10 stratified split preserving class balance.
3. **Baseline model:** Train a simple baseline (e.g., logistic regression on TF-IDF or structural features) to establish a performance floor.
4. **Transformer fine-tuning:** Fine-tune RoBERTa (or a distilled variant) with a classification head on the binary task.
5. **Evaluation:** Report accuracy and macro F1; analyze misclassified examples.
6. **Stretch goal:** Extend to multi-label classification over hack subcategories.

**Potential challenges:**
- Long sequences exceeding model context windows
- Small dataset size (517 examples) — risk of overfitting
- Ensuring the model learns meaningful patterns rather than surface-level shortcuts

**Open questions for TF:**
- Should we prioritize truncation strategies (head/tail/middle) or summarization before feeding to the Transformer?
- Is there value in combining structural features with Transformer embeddings (hybrid approach)?